# Macquarie Precinct — Data Center Suitability Analysis

This notebook performs a spatial suitability analysis to evaluate candidate developable zones within the Macquarie Coal Complex Transformation Precinct for a new data center. It utilizes **Wherobots Spatial SQL (Apache Sedona)** to query, join, and analyze spatial datasets including boundaries, energy lines, transport networks, and waterways.

In [ ]:
# 1. Environment Setup
%pip install --quiet geopandas pyproj requests pandas matplotlib shapely wherobots

from sedona.spark import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, min as spark_min

print("Initializing SedonaContext...")
spark = SedonaContext.create(SedonaContext.builder().getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Sedona Context ready.")

## 2. Spatial SQL Suitability Queries

We query the Havasu tables from `org_catalog.fgsdb` to evaluate each developable zone's size, proximity to high-voltage transmission lines (`macquarie_energy_infrastructure`), and proximity to active railways (`macquarie_rail_network`).

In [ ]:
# Query 1: Calculate Developable Zones Area (in Hectares) and geometry
zones_query = """
    SELECT 
        precinct_key,
        ST_Area(net_developable_geom) / 1e4 AS area_ha,
        ST_AsText(net_developable_geom) as geometry
    FROM org_catalog.fgsdb.macquarie_net_developable_zones
"""
zones_sdf = spark.sql(zones_query)
zones_sdf.show()

In [ ]:
# Query 2: Calculate minimum distance to nearest high-voltage electricity line
# Note: macquarie_energy_infrastructure is in EPSG:4326, we transform it to EPSG:7856 to match the developable zones target CRS
power_query = """
    SELECT 
        z.precinct_key,
        MIN(ST_Distance(z.net_developable_geom, ST_Transform(e.geometry, 'EPSG:4326', 'EPSG:7856'))) AS dist_to_power_m
    FROM org_catalog.fgsdb.macquarie_net_developable_zones z
    CROSS JOIN org_catalog.fgsdb.macquarie_energy_infrastructure e
    GROUP BY z.precinct_key
"""
power_sdf = spark.sql(power_query)
power_sdf.show()

In [ ]:
# Query 3: Calculate minimum distance to nearest active railway line (already in EPSG:7856)
rail_query = """
    SELECT 
        z.precinct_key,
        MIN(ST_Distance(z.net_developable_geom, r.geometry)) AS dist_to_rail_m
    FROM org_catalog.fgsdb.macquarie_net_developable_zones z
    CROSS JOIN org_catalog.fgsdb.macquarie_rail_network r
    GROUP BY z.precinct_key
"""
rail_sdf = spark.sql(rail_query)
rail_sdf.show()

## 4. Interactive Kepler.gl Visualization Map

We use Wherobots' native `wherobots.viz` package to render our spatial layers interactively on a map directly inside the notebook. Note: Geometries must be in WGS84 (EPSG:4326) coordinates to overlay correctly on the base map.

In [ ]:
import wherobots.viz

# 1. Helper function to load spatial queries as GeoPandas GDFs in EPSG:4326
def df_to_gdf_wgs84(df, geom_col="geometry"):
    pdf = df.toPandas()
    if pdf.empty:
        return gpd.GeoDataFrame()
    if geom_col in pdf.columns:
        geoms = pdf[geom_col].apply(lambda g: wkt.loads(g) if g else None)
        if geom_col != "geometry":
            pdf = pdf.drop(columns=[geom_col])
        pdf["geometry"] = geoms
    # Note: net_developable is in EPSG:7856, so we load in 7856 and project to 4326
    gdf = gpd.GeoDataFrame(pdf, geometry="geometry", crs="EPSG:7856").to_crs("EPSG:4326")
    return gdf

# 2. Load layers
net_developable_gdf = df_to_gdf_wgs84(spark.sql("SELECT ST_AsText(net_developable_geom) as geometry, precinct_key FROM org_catalog.fgsdb.macquarie_net_developable_zones"))
energy_gdf = df_to_gdf(spark.sql("SELECT ST_AsText(geometry) as geometry FROM org_catalog.fgsdb.macquarie_energy_infrastructure"))

if not net_developable_gdf.empty:
    net_developable_gdf = net_developable_gdf.merge(metrics_df, on="precinct_key")
    # Plot interactive map colored by suitability score
    wherobots.viz.plot(net_developable_gdf, color_by="suitability_score")
else:
    print("No developable zones found.")

In [ ]:
# 3. Suitability Scoring & Analysis
import pandas as pd
import geopandas as gpd
from shapely import wkt

# Combine spatial metrics into a single Pandas dataframe
zones_pdf = zones_sdf.toPandas()
power_pdf = power_sdf.toPandas()
rail_pdf = rail_sdf.toPandas()

metrics_df = zones_pdf.merge(power_pdf, on="precinct_key").merge(rail_pdf, on="precinct_key")

# Define scoring functions
def score_power(dist):
    # Closer to high-voltage lines is better (ideal < 250m)
    if dist <= 250:
        return 100
    elif dist >= 2000:
        return 0
    else:
        return 100 - ((dist - 250) / 1750) * 100

def score_size(area):
    # Larger parcels are better for data centers (ideal > 15 ha)
    if area >= 15:
        return 100
    elif area < 3:
        return 0
    else:
        return ((area - 3) / 12) * 100

metrics_df["power_score"] = metrics_df["dist_to_power_m"].apply(score_power)
metrics_df["size_score"] = metrics_df["area_ha"].apply(score_size)

# Weighted Suitability Rank (60% Power, 40% Size)
metrics_df["suitability_score"] = (metrics_df["power_score"] * 0.6) + (metrics_df["size_score"] * 0.4)
metrics_df = metrics_df.sort_values(by="suitability_score", ascending=False)

print("=== Data Center Suitability Analysis ===")
print(metrics_df[["precinct_key", "area_ha", "dist_to_power_m", "power_score", "size_score", "suitability_score"]])